# Geo-FNO nuclear model test

This notebook evaluates the trained nuclear Geo-FNO models on the held-out test split, prints test `rel_RMSE`, and saves prediction/truth/error plots for a few test slices.

`rel_RMSE` here is `sqrt(mean((pred - target)^2) / mean(target^2))`, averaged over test batches. It is a relative RMS error, so `1e-3` is roughly a `0.1%` RMS error relative to the RMS temperature scale, not exactly the mean absolute pointwise error relative to the arithmetic mean temperature.

In [ ]:
from __future__ import annotations

import json
import sys
from dataclasses import dataclass
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset

THIS_DIR = Path.cwd()
PARENT_DIR = THIS_DIR.parent
if str(PARENT_DIR) not in sys.path:
    sys.path.insert(0, str(PARENT_DIR))

# geo_FNO_def imports utilities3, which imports scipy.io for MAT-file helpers.
# This test notebook only uses HDF5 data; the tiny stub keeps inference usable
# in lightweight kernels that do not have scipy installed.
try:
    import scipy.io  # noqa: F401
except ModuleNotFoundError:
    import types
    scipy_stub = types.ModuleType('scipy')
    scipy_io_stub = types.ModuleType('scipy.io')
    scipy_io_stub.loadmat = lambda *args, **kwargs: (_ for _ in ()).throw(ModuleNotFoundError('scipy is required for MAT files'))
    scipy_stub.io = scipy_io_stub
    sys.modules.setdefault('scipy', scipy_stub)
    sys.modules.setdefault('scipy.io', scipy_io_stub)

from geo_fno_def_patched import FNO2d, IPHI

DATA_PATH = Path('/scratch/ROM_datasets_nico/Data_Single_Stack/simulation_data_complete.h5')
MODEL_ROOT = Path('/scratch/mnhagen/models/nuclear_dataset')
LOCAL_CHECKPOINT_DIR = THIS_DIR / 'checkpoints'
PREDICTION_DIR = THIS_DIR / 'geo_fno_test_predictions'
BATCH_SIZE = 4
N_PLOTS_PER_MODEL = 3
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

DEFAULT_VAL_TEMPERATURES = [300.0, 360.0, 420.0, 480.0, 540.0, 600.0, 660.0, 720.0, 780.0, 840.0, 900.0, 960.0]
DEFAULT_TEST_TEMPERATURES = [330.0, 390.0, 450.0, 510.0, 570.0, 630.0, 690.0, 750.0, 810.0, 870.0, 930.0, 990.0]
DEFAULT_VAL_Z_LEVELS = [20.0, 320.0]
DEFAULT_TEST_Z_LEVELS = [30.0, 280.0]


@dataclass(frozen=True)
class NuclearSample:
    z_value: float
    temperature_value: float
    temperature_key: str


@dataclass
class ModelSpec:
    label: str
    fno_path: Path
    iphi_path: Path
    feature_names: list[str]
    meta_path: Path | None = None
    meta: dict | None = None


def load_state_dict(path: Path) -> dict[str, torch.Tensor]:
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')


def relative_rmse(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    mse = torch.mean((pred - target) ** 2)
    ref = torch.mean(target ** 2)
    return torch.sqrt(mse / torch.clamp(ref, min=eps))


def safe_name(name: str) -> str:
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in name)


def nearest_levels(available: list[float], requested: list[float]) -> list[float]:
    return [float(min(available, key=lambda z: abs(z - float(z_req)))) for z_req in requested]


class NuclearSliceGeoFNODataset(Dataset):
    def __init__(self, h5_path: Path, feature_names: list[str], split_config: dict | None = None):
        super().__init__()
        self.h5_path = Path(h5_path)
        self.feature_names = list(feature_names)
        self.f = h5py.File(self.h5_path, 'r')
        self.coords = np.array(self.f['physical_field/coordinates'], dtype=np.float32)
        self.temp_group = self.f['physical_field/temperatures']
        self.temperature_values = sorted(float(k.split('_')[1]) for k in self.temp_group.keys())
        self.z_levels = sorted(float(z) for z in np.unique(self.coords[:, 2]))
        self.z_mean = float(np.mean(self.z_levels))
        self.z_std = float(np.std(self.z_levels))
        self.temperature_mean = float(np.mean(self.temperature_values))
        self.temperature_std = float(np.std(self.temperature_values))
        self.z_min = float(min(self.z_levels))
        self.z_max = float(max(self.z_levels))
        self.Lz = max(self.z_max - self.z_min, 1e-6)

        self.xy_by_z = {}
        self.mask_by_z = {}
        self.center_by_z = {}
        z_coords = self.coords[:, 2]
        for z_value in self.z_levels:
            mask = np.isclose(z_coords, z_value, atol=1e-8)
            xy = self.coords[mask, :2].astype(np.float32)
            self.mask_by_z[z_value] = mask
            self.xy_by_z[z_value] = xy
            self.center_by_z[z_value] = xy.mean(axis=0).astype(np.float32)

        self.xy_min = self.coords[:, :2].min(axis=0).astype(np.float32)
        self.xy_max = self.coords[:, :2].max(axis=0).astype(np.float32)
        self.samples = [
            NuclearSample(z_value=z_value, temperature_value=temp_value, temperature_key=f'SS_{int(temp_value)}')
            for z_value in self.z_levels
            for temp_value in self.temperature_values
        ]

        split_config = split_config or {}
        self.val_temperatures = [float(v) for v in split_config.get('val_temperatures', DEFAULT_VAL_TEMPERATURES)]
        self.test_temperatures = [float(v) for v in split_config.get('test_temperatures', DEFAULT_TEST_TEMPERATURES)]
        self.val_z_levels = nearest_levels(self.z_levels, split_config.get('val_z_levels', DEFAULT_VAL_Z_LEVELS))
        self.test_z_levels = nearest_levels(self.z_levels, split_config.get('test_z_levels', DEFAULT_TEST_Z_LEVELS))

    def __len__(self) -> int:
        return len(self.samples)

    def normalize_z(self, z_value: float) -> float:
        return (float(z_value) - self.z_mean) / max(self.z_std, 1e-6)

    def normalize_temperature(self, temperature_value: float) -> float:
        return (float(temperature_value) - self.temperature_mean) / max(self.temperature_std, 1e-6)

    def feature_value(self, feature_name: str, z_value: float, temperature_value: float) -> float:
        z = float(z_value)
        temp = float(temperature_value)
        z_hat = self.normalize_z(z)
        temp_hat = self.normalize_temperature(temp)
        z_ax = (z - self.z_min) / self.Lz
        sin1 = float(np.sin(np.pi * z_ax))
        cos1 = float(np.cos(np.pi * z_ax))
        sin2 = float(np.sin(2.0 * np.pi * z_ax))
        cos2 = float(np.cos(2.0 * np.pi * z_ax))
        name = feature_name.replace(' ', '')
        if name in {'z', 'raw_z'}:
            return z
        if name in {'T', 'temperature', 'BC_temperature'}:
            return temp
        if name in {'z_hat', 'normalized_z'}:
            return z_hat
        if name in {'T_hat', 'temperature_hat'}:
            return temp_hat
        if name == 'z^2':
            return z * z
        if name == 'T^2':
            return temp * temp
        if name == 'z*T':
            return z * temp
        if name == 'z_hat^2':
            return z_hat * z_hat
        if name == 'T_hat^2':
            return temp_hat * temp_hat
        if name == 'z_hat*T_hat':
            return z_hat * temp_hat
        if name == 'sin(pi*(z-z_min)/Lz)':
            return sin1
        if name == 'T*sin(pi*(z-z_min)/Lz)':
            return temp * sin1
        if name == 'cos(pi*(z-z_min)/Lz)':
            return cos1
        if name == 'T*cos(pi*(z-z_min)/Lz)':
            return temp * cos1
        if name == 'sin(2*pi*(z-z_min)/Lz)':
            return sin2
        if name == 'T*sin(2*pi*(z-z_min)/Lz)':
            return temp * sin2
        if name == 'cos(2*pi*(z-z_min)/Lz)':
            return cos2
        if name == 'T*cos(2*pi*(z-z_min)/Lz)':
            return temp * cos2
        raise ValueError(f'Unknown feature name: {feature_name!r}')

    def __getitem__(self, idx: int):
        sample = self.samples[idx]
        xy = self.xy_by_z[sample.z_value]
        mask = self.mask_by_z[sample.z_value]
        field = np.array(self.temp_group[sample.temperature_key], dtype=np.float32)[mask]
        features = [self.feature_value(name, sample.z_value, sample.temperature_value) for name in self.feature_names]
        u_in = np.repeat(np.asarray(features, dtype=np.float32)[None, :], xy.shape[0], axis=0)
        u_out = field.reshape(-1, 1).astype(np.float32)
        code = np.zeros((42,), dtype=np.float32)
        code[0:2] = self.center_by_z[sample.z_value]
        return torch.from_numpy(xy), torch.from_numpy(u_in), torch.from_numpy(u_out), torch.from_numpy(code), idx

    def build_default_splits(self) -> tuple[list[int], list[int], list[int]]:
        val_pairs = {(z, t) for z in self.val_z_levels for t in self.val_temperatures}
        test_pairs = {(z, t) for z in self.test_z_levels for t in self.test_temperatures}
        train_idx, val_idx, test_idx = [], [], []
        for idx, sample in enumerate(self.samples):
            pair = (sample.z_value, sample.temperature_value)
            if pair in test_pairs:
                test_idx.append(idx)
            elif pair in val_pairs:
                val_idx.append(idx)
            else:
                train_idx.append(idx)
        return train_idx, val_idx, test_idx


def resolve_meta_checkpoint(meta_path: Path, meta: dict, kind: str) -> Path:
    key = 'model_path' if kind == 'fno' else 'iphi_path'
    direct = Path(meta.get(key, ''))
    if direct.exists():
        return direct
    suffix = '_fno.pt' if kind == 'fno' else '_iphi.pt'
    sibling = meta_path.with_name(meta_path.name.replace('_meta.json', suffix))
    if sibling.exists():
        return sibling
    return direct


def specs_from_metadata() -> list[ModelSpec]:
    specs = []
    for meta_path in sorted(MODEL_ROOT.glob('**/*_meta.json')):
        with open(meta_path) as f:
            meta = json.load(f)
        feature_names = meta.get('feature_config', {}).get('input_features')
        if not feature_names:
            continue
        fno_path = resolve_meta_checkpoint(meta_path, meta, 'fno')
        iphi_path = resolve_meta_checkpoint(meta_path, meta, 'iphi')
        if fno_path.exists() and iphi_path.exists():
            label = f'saved/{meta_path.relative_to(MODEL_ROOT).with_suffix("")}'
            label = label.replace('_meta', '')
            specs.append(ModelSpec(label, fno_path, iphi_path, list(feature_names), meta_path, meta))
        else:
            print(f'Skipping missing saved model from {meta_path}: {fno_path}, {iphi_path}')
    return specs


def local_checkpoint_specs() -> list[ModelSpec]:
    local = [
        ('local/vanilla_ep200', 'geofno_nuclear_ep200.pt', 'iphi_nuclear_ep200.pt', ['z', 'T']),
        ('local/quadratic_ep200', 'geofno_nuclear_quadratic_ep200.pt', 'iphi_nuclear_quadratic_ep200.pt', ['z_hat', 'T_hat', 'z_hat^2', 'T_hat^2', 'z_hat*T_hat']),
        ('local/interaction_ep100', 'geofno_nuclear_interaction_ep100.pt', 'iphi_nuclear_interaction_ep100.pt', ['z_hat', 'T_hat', 'z_hat*T_hat']),
        ('local/physical_ep100', 'geofno_nuclear_physical_ep100.pt', 'iphi_nuclear_physical_ep100.pt', ['z', 'T', 'sin(pi*(z-z_min)/Lz)', 'T*sin(pi*(z-z_min)/Lz)', 'cos(pi*(z-z_min)/Lz)']),
        ('local/normalized_ep0', 'geofno_nuclear_normalized_ep0.pt', 'iphi_nuclear_normalized_ep0.pt', ['z_hat', 'T_hat']),
    ]
    specs = []
    for label, fno_name, iphi_name, features in local:
        fno_path = LOCAL_CHECKPOINT_DIR / fno_name
        iphi_path = LOCAL_CHECKPOINT_DIR / iphi_name
        if fno_path.exists() and iphi_path.exists():
            specs.append(ModelSpec(label, fno_path, iphi_path, features))
    return specs


def model_input_channels(state_dict: dict[str, torch.Tensor]) -> int:
    return int(state_dict['fc0.weight'].shape[1])


def load_model(spec: ModelSpec, device: str) -> tuple[FNO2d, IPHI]:
    fno_state = load_state_dict(spec.fno_path)
    in_channels = model_input_channels(fno_state)
    if len(spec.feature_names) != in_channels:
        raise ValueError(f'{spec.label}: feature list has {len(spec.feature_names)} entries, but checkpoint expects {in_channels}')
    config = (spec.meta or {}).get('model_config', {})
    xy_min = np.asarray((spec.meta or {}).get('xy_min', [-18.0, -20.784608840942383]), dtype=np.float32)
    xy_max = np.asarray((spec.meta or {}).get('xy_max', [18.0, 20.784608840942383]), dtype=np.float32)
    model = FNO2d(
        config.get('modes1', 16),
        config.get('modes2', 16),
        config.get('width', 32),
        in_channels=in_channels,
        out_channels=1,
        is_mesh=False,
        s1=config.get('s1', 64),
        s2=config.get('s2', 64),
        L=[float(xy_max[0] - xy_min[0]), float(xy_max[1] - xy_min[1])],
    ).to(device)
    iphi = IPHI(width=config.get('iphi_width', 32), device=device).to(device)
    model.load_state_dict(fno_state, strict=True)
    iphi.load_state_dict(load_state_dict(spec.iphi_path), strict=True)
    model.eval()
    iphi.eval()
    return model, iphi


def evaluate_and_collect(model: FNO2d, iphi: IPHI, loader: DataLoader, ds: NuclearSliceGeoFNODataset, device: str):
    mse_loss = nn.MSELoss()
    total_mse = 0.0
    total_rel = 0.0
    batches = 0
    examples = []
    with torch.no_grad():
        for pos, u_in, u_out, code, sample_idx in loader:
            pos = pos.to(device)
            u_in = u_in.to(device)
            u_out = u_out.to(device)
            code = code.to(device)
            pred = model(u_in, code=code, x_in=pos, x_out=pos, iphi=iphi)
            total_mse += float(mse_loss(pred, u_out).item())
            total_rel += float(relative_rmse(pred, u_out).item())
            batches += 1
            if len(examples) < N_PLOTS_PER_MODEL:
                for b in range(pred.shape[0]):
                    idx = int(sample_idx[b].item())
                    sample = ds.samples[idx]
                    examples.append({
                        'sample': sample,
                        'pos': pos[b].detach().cpu().numpy(),
                        'pred': pred[b, :, 0].detach().cpu().numpy(),
                        'target': u_out[b, :, 0].detach().cpu().numpy(),
                        'rel_rmse': float(relative_rmse(pred[b:b+1], u_out[b:b+1]).item()),
                    })
                    if len(examples) >= N_PLOTS_PER_MODEL:
                        break
    return total_mse / max(1, batches), total_rel / max(1, batches), examples


def plot_examples(label: str, examples: list[dict]) -> None:
    out_dir = PREDICTION_DIR / safe_name(label)
    out_dir.mkdir(parents=True, exist_ok=True)
    for i, ex in enumerate(examples):
        pos = ex['pos']
        pred = ex['pred']
        target = ex['target']
        err = pred - target
        sample = ex['sample']
        vmin = float(min(target.min(), pred.min()))
        vmax = float(max(target.max(), pred.max()))
        emax = float(np.max(np.abs(err)))
        fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
        panels = [('truth', target, vmin, vmax, 'inferno'), ('prediction', pred, vmin, vmax, 'inferno'), ('error', err, -emax, emax, 'coolwarm')]
        for ax, (title, values, lo, hi, cmap) in zip(axes, panels):
            sc = ax.scatter(pos[:, 0], pos[:, 1], c=values, s=5, cmap=cmap, vmin=lo, vmax=hi)
            ax.set_title(title)
            ax.set_aspect('equal')
            ax.set_xlabel('x')
            ax.set_ylabel('y')
            fig.colorbar(sc, ax=ax, shrink=0.8)
        fig.suptitle(f'{label} | z={sample.z_value:g}, T_bc={sample.temperature_value:g}, slice rel_RMSE={ex["rel_rmse"]:.3e}')
        path = out_dir / f'prediction_{i:02d}_z{sample.z_value:g}_T{sample.temperature_value:g}.png'
        fig.savefig(path, dpi=180)
        plt.show()
        plt.close(fig)


In [ ]:
print('Device:', DEVICE)
print('Data:', DATA_PATH)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

specs = specs_from_metadata() + local_checkpoint_specs()
print(f'Found {len(specs)} nuclear model specs')
for spec in specs:
    print('-', spec.label, '| features:', spec.feature_names)

results = []
for spec in specs:
    print('\n===', spec.label, '===')
    try:
        ds = NuclearSliceGeoFNODataset(DATA_PATH, spec.feature_names, (spec.meta or {}).get('split_config'))
        _, _, test_idx = ds.build_default_splits()
        test_loader = DataLoader(Subset(ds, test_idx), batch_size=BATCH_SIZE, shuffle=False)
        model, iphi = load_model(spec, DEVICE)
        test_mse, test_rel_rmse, examples = evaluate_and_collect(model, iphi, test_loader, ds, DEVICE)
        print(f'Test rel_RMSE: {test_rel_rmse:.6e} | Test MSE: {test_mse:.6e} | n_test={len(test_idx)}')
        if spec.meta and 'test_metrics' in spec.meta:
            print('Saved-meta rel_RMSE:', f"{spec.meta['test_metrics'].get('rel_l2', float('nan')):.6e}")
        plot_examples(spec.label, examples)
        results.append({'label': spec.label, 'test_rel_RMSE': test_rel_rmse, 'test_MSE': test_mse, 'n_test': len(test_idx)})
    except Exception as exc:
        print('SKIPPED:', exc)

print('\nSummary')
for row in sorted(results, key=lambda r: r['test_rel_RMSE']):
    print(f"{row['label']:<72} rel_RMSE={row['test_rel_RMSE']:.6e} MSE={row['test_MSE']:.6e} n={row['n_test']}")
print('\nPrediction figures saved under:', PREDICTION_DIR)
